In [2]:
import pandas as pd
import re


file_path = '../../datasets/gekuerzt_bereinigt.csv'

df = pd.read_csv(file_path, dtype=str)
df = pd.DataFrame(df)

#print(df.columns.to_list())

df.drop(['link', 'source', 'Unnamed: 0'], axis=1, inplace=True)

#def remove_columns(df):
#    df.drop(['link', 'source'], axis=1, inplace=True)
#    return df

print(df.head(3))
print("Total number of character:", len(df))
print(df[:99])



                                               title  \
0                     Vegetable Frittata with Cheese   
1  Grilled Clams with Spaghetti, Prosciutto, and ...   
2                          Spiced up Vegetable Pasta   

                                         ingredients  \
0  ["8 large eggs slightly beaten", "1 tablespoon...   
1  ["1/2 cup extra-virgin olive oil", "4 garlic c...   
2  ["14 cup bell pepper, finely chopped (yellow, ...   

                                          directions  \
0  ["In a medium bowl combine eggs and basil; set...   
1  ["Heat oil in heavy large pot over medium-high...   
2  ["Bring a large pot of salted water to a rolli...   

                                                 NER  
0  ["eggs", "basil", "olive oil", "whole kernel c...  
1  ["extra-virgin olive oil", "garlic", "red pepp...  
2  ["bell pepper", "tomatoes", "eggplant", "garli...  
Total number of character: 100000
                                                title  \
0             

Tokenizer: leerzeichen -> cutten -> array aus allen Wörtern; aus jedem Wort eine ID und aus jeder ID ein Vektor

In [3]:
def format_csv(row):
    title = str(row['title'])
    ingredients = str(row['ingredients']).replace('[', '').replace(']', '').replace("'", "").replace('"', '')
    directions = str(row['directions']).replace('[', '').replace(']', '').replace("'", "").replace('"', '')
    return f"Recepie: {title}\nIngredients:{ingredients}\nDirections:{directions}"

raw_text = "".join(df.apply(format_csv, axis=1))
print(f"Der gesamte Datensatz wurde in einen String umgewandelt.")
print(f"Gesamtanzahl der Zeichen: {len(raw_text)}")
print("\nVorschau der ersten 300 Zeichen:")
print(raw_text[:1000])

Der gesamte Datensatz wurde in einen String umgewandelt.
Gesamtanzahl der Zeichen: 98532627

Vorschau der ersten 300 Zeichen:
Recepie: Vegetable Frittata with Cheese
Ingredients:8 large eggs slightly beaten, 1 tablespoon basil snipped fresh, teaspoon dried basil, crushed, 2 tablespoons olive oil, 1 cup whole kernel corn, frozen or cut fresh corn, 1/2 cup zucchini chopped, 13 cup scallions, spring or green onions thinly sliced, 3/4 cup italian plum (roma) tomatoes chopped, 1 x salt and black pepper to taste, 1/2 cup cheddar cheese, reduced-fat shredded, 2 ounces
Directions:In a medium bowl combine eggs and basil; set aside., Heat oil in a large broilerproof skillet; add corn, zucchini, and green onions., Cook and stir for 3 minutes; add tomatoes, salt and pepper to taste., Cook, uncovered, over medium heat about 5 minutes or until vegetables are crisp-tender, stirring occasionally., Pour egg mixture over vegetables in skillet., Cook over medium heat., As mixture sets, run a spatula arou

Tokenizer zerlegt den Text in einzelne teile 

In [9]:
def trokenizer_funktion(wert: str) -> list:
    result = re.split(r'(\s)', wert)

    final = []

    for token in result:
        teile = re.split(r'([,.]|\s)', token)
        final.extend(teile)

    sauber = []
    for item in final:
        if item.strip():
            sauber.append(item)
    
    
    
    return sauber

teilcutting = trokenizer_funktion(raw_text)
print(teilcutting[:500])
print(len(teilcutting))



['Recepie:', 'Vegetable', 'Frittata', 'with', 'Cheese', 'Ingredients:8', 'large', 'eggs', 'slightly', 'beaten', ',', '1', 'tablespoon', 'basil', 'snipped', 'fresh', ',', 'teaspoon', 'dried', 'basil', ',', 'crushed', ',', '2', 'tablespoons', 'olive', 'oil', ',', '1', 'cup', 'whole', 'kernel', 'corn', ',', 'frozen', 'or', 'cut', 'fresh', 'corn', ',', '1/2', 'cup', 'zucchini', 'chopped', ',', '13', 'cup', 'scallions', ',', 'spring', 'or', 'green', 'onions', 'thinly', 'sliced', ',', '3/4', 'cup', 'italian', 'plum', '(roma)', 'tomatoes', 'chopped', ',', '1', 'x', 'salt', 'and', 'black', 'pepper', 'to', 'taste', ',', '1/2', 'cup', 'cheddar', 'cheese', ',', 'reduced-fat', 'shredded', ',', '2', 'ounces', 'Directions:In', 'a', 'medium', 'bowl', 'combine', 'eggs', 'and', 'basil;', 'set', 'aside', '.', ',', 'Heat', 'oil', 'in', 'a', 'large', 'broilerproof', 'skillet;', 'add', 'corn', ',', 'zucchini', ',', 'and', 'green', 'onions', '.', ',', 'Cook', 'and', 'stir', 'for', '3', 'minutes;', 'add', 't

Token in eine ID umschreiben: (encoder)

In [15]:
def erstelle_vokabular(wert: list) -> dict:
    vokabular = {}

    id = 0

    for items in wert:
        if items not in vokabular:
            vokabular[items] = id
            id += 1
    
    return vokabular


def encode(tokens: list, vokabular: dict) -> list:
    return [vokabular[token] for token in tokens]


vokabular = erstelle_vokabular(teilcutting)   
ids = encode(teilcutting, vokabular)  

print(dict(list(vokabular.items())[:10])) 
print(ids[:50])
print(len(vokabular))

{'Recepie:': 0, 'Vegetable': 1, 'Frittata': 2, 'with': 3, 'Cheese': 4, 'Ingredients:8': 5, 'large': 6, 'eggs': 7, 'slightly': 8, 'beaten': 9}
[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 10, 16, 17, 13, 10, 18, 10, 19, 20, 21, 22, 10, 11, 23, 24, 25, 26, 10, 27, 28, 29, 15, 26, 10, 30, 23, 31, 32, 10, 33, 23, 34, 10, 35]
134347


Special Token: wichtige Wörter werden hinzugefügt; damit das Modell anfänge und enden kategorisieren kann
<|endoftext|>; Ende des textes; padding; trennung zwischen zwei texten

In [16]:
def tokens_zu_ids(tokens: list, vokabular: dict) -> list:
    ids = []
    for token in tokens:
        if token in vokabular:
            ids.append(vokabular[token])
        else:
            ids.append(vokabular['<|endoftext|>'])  # unbekannte tokens
    return ids
ids = tokens_zu_ids(teilcutting, vokabular)
print(ids[:50])

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 10, 16, 17, 13, 10, 18, 10, 19, 20, 21, 22, 10, 11, 23, 24, 25, 26, 10, 27, 28, 29, 15, 26, 10, 30, 23, 31, 32, 10, 33, 23, 34, 10, 35]


ID wieder in Tokens umschreiben: (decode)

In [18]:
#mithilfe der KI gelöst
def decode(ids: list) -> str:
    
    umgekehrt = {}
    for token, idx in vokabular.items():
        umgekehrt[idx] = token
    
    tokens = []
    for id in ids:
        tokens.append(umgekehrt[id])
    
    return ' '.join(tokens)


print(decode(ids[:1000]))

Recepie: Vegetable Frittata with Cheese Ingredients:8 large eggs slightly beaten , 1 tablespoon basil snipped fresh , teaspoon dried basil , crushed , 2 tablespoons olive oil , 1 cup whole kernel corn , frozen or cut fresh corn , 1/2 cup zucchini chopped , 13 cup scallions , spring or green onions thinly sliced , 3/4 cup italian plum (roma) tomatoes chopped , 1 x salt and black pepper to taste , 1/2 cup cheddar cheese , reduced-fat shredded , 2 ounces Directions:In a medium bowl combine eggs and basil; set aside . , Heat oil in a large broilerproof skillet; add corn , zucchini , and green onions . , Cook and stir for 3 minutes; add tomatoes , salt and pepper to taste . , Cook , uncovered , over medium heat about 5 minutes or until vegetables are crisp-tender , stirring occasionally . , Pour egg mixture over vegetables in skillet . , Cook over medium heat . , As mixture sets , run a spatula around edge of skillet , lifting egg mixture so uncooked portion flows underneath . , Continue co

In [ ]:
import tiktoken

enc = tiktoken.get_encoding('gpt2')

teilcutting = str(teilcutting)

ids = enc.encode(teilcutting)
print('encode', ids[:100])

text = enc.decode(ids)
print('decode', text[:100])

encode [17816, 3041, 344, 21749, 25, 3256, 705, 26979, 1136, 540, 3256, 705, 6732, 715, 1045, 3256, 705, 4480, 3256, 705, 7376, 2771, 3256, 705, 41222, 25, 23, 3256, 705, 11664, 3256, 705, 33856, 82, 3256, 705, 82, 30945, 3256, 705, 12945, 268, 3256, 46083, 3256, 705, 16, 3256, 705, 83, 2977, 26743, 3256, 705, 12093, 346, 3256, 705, 16184, 3949, 3256, 705, 48797, 3256, 46083, 3256, 705, 660, 5126, 2049, 3256, 705, 67, 2228, 3256, 705, 12093, 346, 3256, 46083, 3256, 705, 6098, 7474, 3256, 46083, 3256, 705, 17, 3256, 705, 83, 2977, 27575, 3256, 705, 349, 425, 3256, 705]
['Recepie:', 'Vegetable', 'Frittata', 'with', 'Cheese', 'Ingredients:8', 'large', 'eggs', 'slightly'


sliding window (Labeling)

In [23]:
def sliding_window(tokens: list, fenster: int) -> tuple:
    inputs = []
    targets = []

    for i in range(len(tokens) - fenster):
        input = tokens[i : i + fenster]    # Fenster ausschneiden
        target = tokens[i + fenster]        # nächstes Token = Zielvariable
        
        inputs.append(input)
        targets.append(target)
    
    return inputs, targets


# Testen:
tokens = [1, 2, 3, 4, 5, 6, 7]
fenster = 3

inputs, targets = sliding_window(tokens, fenster)

for i, (inp, tar) in enumerate(zip(inputs, targets)):
    print(f"Paar {i+1}: Input {inp} → Target {tar}")

Paar 1: Input [1, 2, 3] → Target 4
Paar 2: Input [2, 3, 4] → Target 5
Paar 3: Input [3, 4, 5] → Target 6
Paar 4: Input [4, 5, 6] → Target 7


In [24]:
import torch
from torch.utils.data import Dataset, DataLoader

# Schritt 1 – Dataset Klasse
class TextDataset(Dataset):
    
    def __init__(self, tokens: list, fenster: int):
        self.inputs = []
        self.targets = []
        
        for i in range(len(tokens) - fenster):
            self.inputs.append(tokens[i : i + fenster])
            self.targets.append(tokens[i + fenster])
    
    def __len__(self):
        return len(self.inputs)
    
    def __getitem__(self, idx):
        return (
            torch.tensor(self.inputs[idx]),   # Input als Tensor
            torch.tensor(self.targets[idx])   # Target als Tensor
        )


# Schritt 2 – Dataloader erstellen
dataset = TextDataset(ids, fenster=3)
dataloader = DataLoader(dataset, batch_size=2, shuffle=True)

# Schritt 3 – ersten Batch ausgeben
for inputs, targets in dataloader:
    print(f"Input:  {inputs}")
    print(f"Target: {targets}")
    break  # nur ersten Batch zeigen

Input:  tensor([[4360, 3256,  705],
        [ 705, 3549, 3256]])
Target: tensor([1662,  705])


In [26]:

import torch.nn as nn

# Vokabular Größe: 50257 (GPT-2)
# Embedding Dimension: 256
embedding = nn.Embedding(50257, 256)

# Token ID → Vektor
token_id = torch.tensor([1, 2, 3])
vektor = embedding(token_id)
print(vektor.shape)  # → torch.Size([3, 256])

torch.Size([3, 256])


In [27]:

import torch.nn as nn

vocab_size = len(vokabular)
embed_dim = 256
fenster = 3

# 1. Token Embedding – was ist das Wort?
token_embedding = nn.Embedding(vocab_size, embed_dim)

# 2. Position Embedding – wo steht das Wort?
pos_embedding = nn.Embedding(fenster, embed_dim)

# Zusammen:
for inputs, targets in dataloader:
    token_vektoren = token_embedding(inputs)
    
    positionen = torch.arange(fenster)           # → [0, 1, 2]
    pos_vektoren = pos_embedding(positionen)     # → Position als Vektor
    
    x = token_vektoren + pos_vektoren            # addieren!
    print(f"Input Shape: {inputs.shape}")        # → [2, 3]
    print(f"Output Shape: {x.shape}")            # → [2, 3, 256]
    break

Input Shape: torch.Size([2, 3])
Output Shape: torch.Size([2, 3, 256])
